# 2.6 Python Sets Datatype

**Prerequisites:** 2.5 Python Dictionary Datatype  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a set is, and the three properties that define it
- Why membership testing is O(1) and why that matters
- Creating sets — and why `{}` gives you a dict
- The four set operations, and their mutating twins
- `add`/`update`, `remove`/`discard`/`pop`
- `frozenset` for hashable sets, and set comprehensions
- Deduplication, with and without preserving order

---

## Python Sets:

A **set** is an **unordered** collection of **unique**, **hashable** elements, written in
curly braces `{}` with commas between items.

Three properties, and each one is doing work:

| Property | Consequence |
|---|---|
| **Unordered** | No indexing, no slicing, no `.sort()`. Iteration order is not guaranteed. |
| **Unique** | Duplicates are silently discarded on insertion. |
| **Hashable elements** | Elements must be immutable — `int`, `str`, `tuple`, `frozenset`. Not `list`, `dict` or `set`. |

The **set itself is mutable** (you can add and remove), even though its *elements* must be
immutable. `frozenset` is the immutable version.

### What is a set actually for?

Two things, and it is dramatically better than a list at both:

1. **"Have I seen this before?"** — membership testing is **O(1)** instead of O(n).
2. **"What do these two collections have in common?"** — union, intersection and difference
   are single operators instead of nested loops.

**Analogy:** a list is a queue of people in order, duplicates and all. A set is the
*guest list* — each name appears once, the order is meaningless, and the only question you
ever ask is "is this person on it?"

**Real-world use case:** deduplicating email addresses, tracking which URLs a crawler has
already visited, computing which permissions changed between two roles, finding tags shared
by two articles.

> A Python set is a direct implementation of a mathematical set — which is why the operators
> `|`, `&`, `-` and `^` read exactly like the notation from set theory.

### Create Empty Sets

In [ ]:
s0 = set()
print(type(s0), len(s0))

> ### ⚠️ `{}` is an empty **dict**, not an empty set
>
> ```python
> type({})       # <class 'dict'>
> type(set())    # <class 'set'>
> ```
>
> Dictionaries claimed the curly braces first, so there is **no literal syntax for an empty
> set**. You must call `set()`.
>
> Once a set has elements the ambiguity disappears — `{1, 2, 3}` is unmistakably a set,
> because there are no colons.

### Create a Non-Empty Sets

In [ ]:
s0 = {3, 4.5, 'hi', 7}

In [ ]:
# User defined set:
s1 = []
ip = input("Enter csv of set: ").split(',')
for i in ip:
    if i.isalpha():
        s1.append(i)
    elif i.isdigit():
        s1.append(int(i))
    else:
        s1.append(float(i))
print(set(s1))

In [ ]:
# print(s0[1]) # give error since 'set' object is not subscriptable

Hence neither indexing, slicing nor assignment is possible in sets

In [ ]:
print(3 in s0)     # membership operation
print('Hey' in s0) 

> **Why membership testing is the point.** A set stores its elements in a **hash table**.
> Checking `x in s` computes one hash and looks in one place — **O(1)**, regardless of size.
>
> Checking `x in some_list` walks the list from the start — **O(n)**.
>
> For 10 elements nobody notices. For 200,000, the difference is roughly four orders of
> magnitude. There's a benchmark at the end of this notebook.

In [ ]:
s2 = {3, 'hi', 5, 12, 'hey'}

### Set operations

This is where sets earn their place. The four core operations map directly onto the Venn
diagrams from school mathematics:

| Operation | Operator | Meaning | Venn |
|---|---|---|---|
| **Union** | `a \| b` | In either | Both circles |
| **Intersection** | `a & b` | In both | The overlap |
| **Difference** | `a - b` | In `a` but not `b` | Left circle minus overlap |
| **Symmetric difference** | `a ^ b` | In exactly one | Both circles minus overlap |

And the comparison operators test containment rather than magnitude:

| Test | Operator | Meaning |
|---|---|---|
| Subset | `a <= b` | Every element of `a` is in `b` |
| Proper subset | `a < b` | Subset, and not equal |
| Superset | `a >= b` | Every element of `b` is in `a` |
| Disjoint | `a.isdisjoint(b)` | No elements in common |

**Real-world use case:** "which permissions does this user have that the role doesn't?"
is `user_perms - role_perms`. "Which tags do these two articles share?" is `a & b`. Written
with lists and loops, both are ten lines and O(n·m); with sets they are one line and O(n).

In [ ]:
print(s0.union(s2))  # new set with elements from both s0 and s2
print(s0|s2)

In [ ]:
print(s0.intersection(s2))  # new set with elements common to s0 and s2
print(s0 & s2) 

In [ ]:
print(s0.difference(s2))  # new set with elements in s0 but not in s2
print(s0 - s2)

In [ ]:
print(s0.symmetric_difference(s2))  # new set with elements in either s0 or s1 but not both
print(s0 ^ s2)

In [ ]:
print(s0.copy())  # new set with a shallow copy of s0

In [ ]:
s3 = {'hi', 7}
print(s3.issubset(s0))   # test whether every element in s3 is in s0
print(s3 <= s0)

In [ ]:
print(s0.issuperset(s3))   # test whether every element in s2 is in s0
print(s0 >= s3)

In [ ]:
s0.add('Hello')   # Adds the item x to set if it is not already present in the set.
print(s0)

In [ ]:
s4 = {101, 'Python'}   # return set s0 with elements added from s4
s0.update(s4)
print(s0)

### Mutating vs non-mutating: every operation comes in two flavours

This is the single most common source of confusion with sets. Each operation has a form
that **returns a new set** and a form that **modifies in place and returns `None`**.

| Operation | Operator (new set) | Method (new set) | In-place |
|---|---|---|---|
| Union | `a \| b` | `a.union(b)` | `a \|= b` / `a.update(b)` |
| Intersection | `a & b` | `a.intersection(b)` | `a &= b` / `a.intersection_update(b)` |
| Difference | `a - b` | `a.difference(b)` | `a -= b` / `a.difference_update(b)` |
| Symmetric difference | `a ^ b` | `a.symmetric_difference(b)` | `a ^= b` / `a.symmetric_difference_update(b)` |

**One real difference between the operator and method forms:** operators require *both*
operands to be sets; the methods accept **any iterable**. `{1,2}.union([3,4])` works;
`{1,2} | [3,4]` raises `TypeError`.

### Adding and removing

| Method | Effect | If absent |
|---|---|---|
| `add(x)` | Add one element | — |
| `update(iterable)` | Add many | — |
| `remove(x)` | Remove one | Raises `KeyError` |
| `discard(x)` | Remove one | Silently does nothing |
| `pop()` | Remove an **arbitrary** element | Raises `KeyError` if empty |
| `clear()` | Remove everything | — |

In [ ]:
a = {1, 2, 3}
b = {3, 4, 5}

# Non-mutating: return a NEW set, leave both operands alone
print("a | b :", a | b, "| a is still", a)

# Mutating: change `a` in place, return None
result = a.update(b)
print("\na.update(b) returned:", result, " <- None")
print("a is now            :", a)

# Every operator has a mutating twin
x = {1, 2, 3, 4}
x &= {2, 3, 9}          # intersection_update
print("\nafter &= :", x)

x = {1, 2, 3, 4}
x -= {2, 4}             # difference_update
print("after -= :", x)

x = {1, 2, 3, 4}
x ^= {3, 4, 5}          # symmetric_difference_update
print("after ^= :", x)

# Operators require BOTH sides to be sets; methods accept any iterable
print("\nmethod with a list :", {1, 2}.union([3, 4]))
try:
    {1, 2} | [3, 4]
except TypeError as exc:
    print("operator with a list:", exc)

# add() one element vs update() many
s = {1}
s.add(2)
print("\nafter add(2)      :", s)
s.update([3, 4], {5})
print("after update(...) :", s)

# ⚠️ add() a string adds the WHOLE string; update() adds each character
s1, s2 = set(), set()
s1.add("abc")
s2.update("abc")
print("\nadd('abc')   ->", s1)
print("update('abc')->", s2, " <- iterated the string!")

In [ ]:
s0.remove('Hello')    # remove x from set s0; raises KeyError if not present
print(s0)

In [ ]:
s0.discard('hi')    # removes x from set s0 if present
print(s0)

In [ ]:
s0.pop()   # remove and return an arbitrary element from s0
print(s0)

In [ ]:
s0.clear()   # remove all elements from set s0

---

## `frozenset` — the immutable set

A `set` is mutable, and therefore **unhashable** — which means a set cannot be a dictionary
key, and cannot be an element of another set.

`frozenset` is the immutable counterpart. Same operations minus the mutating ones, plus
hashability. It is to `set` what `tuple` is to `list`.

## Set comprehensions

Sets get the same comprehension syntax as lists and dicts — braces with no colon:

```
{expression for item in iterable if condition}
```

(Full treatment of comprehensions is in **03 Flow Control Statement**.)

In [ ]:
# frozenset: an immutable set. Hashable, so it can be a key or an element.
fs = frozenset([1, 2, 3])
print("frozenset:", fs, type(fs))

# All the read operations work
print("2 in fs        :", 2 in fs)
print("fs | {4}       :", fs | {4})
print("fs & {2, 3, 9} :", fs & {2, 3, 9})

# None of the mutating ones do
try:
    fs.add(4)
except AttributeError as exc:
    print("\nfs.add(4):", exc)

# A regular set cannot go inside another set - it is unhashable
try:
    {{1, 2}, {3, 4}}
except TypeError as exc:
    print("set inside a set:", exc)

# frozenset can
print("\nset of frozensets:", {frozenset([1, 2]), frozenset([3, 4])})

# The real payoff: unordered pairs that compare equal regardless of order
pairs = {frozenset(["alice", "bob"]), frozenset(["bob", "alice"])}
print("dedup unordered pairs:", pairs, "| count:", len(pairs))

# frozenset as a dict key
routes = {frozenset(["Delhi", "Mumbai"]): 1150}
print("\nroute lookup (either order):", routes[frozenset(["Mumbai", "Delhi"])])


# ---- Set comprehension ----
sentence = "the quick brown fox jumps over the lazy dog"

lengths = {len(word) for word in sentence.split()}
print("\ndistinct word lengths:", lengths)

vowels_used = {ch for ch in sentence if ch in "aeiou"}
print("vowels used          :", vowels_used)

# Compare the three brace-based comprehensions
print("\nset  :", {x % 3 for x in range(10)})
print("dict :", {x: x % 3 for x in range(5)})
print("list :", [x % 3 for x in range(10)])

In [ ]:
import time

# Sets buy you O(1) membership testing. That is the headline feature.
big_list = list(range(200_000))
big_set = set(big_list)
targets = [199_999, 150_000, 99_999]

start = time.perf_counter()
for t in targets:
    _ = t in big_list
list_time = time.perf_counter() - start

start = time.perf_counter()
for t in targets:
    _ = t in big_set
set_time = time.perf_counter() - start

print(f"list membership: {list_time * 1000:8.3f} ms   (O(n) scan)")
print(f"set  membership: {set_time * 1000:8.3f} ms   (O(1) hash)")
print(f"roughly {list_time / set_time:,.0f}x faster")

# ---- The classic real-world use: deduplication ----
emails = ["a@x.com", "b@x.com", "a@x.com", "c@x.com", "b@x.com"]

print("\nunique (order lost)    :", set(emails))
print("unique (order kept)    :", list(dict.fromkeys(emails)))

# ---- Finding what changed between two states ----
yesterday = {"alice", "bob", "carol"}
today = {"bob", "carol", "dave"}

print("\njoined :", today - yesterday)
print("left   :", yesterday - today)
print("stayed :", today & yesterday)
print("churned:", today ^ yesterday)

---

## Common Mistakes & Pitfalls

1. **`{}` creates an empty dict, not an empty set.** Use `set()`. There is no set literal for the empty set — the braces were taken first.
2. **Expecting sets to keep order.** They do not, and unlike dicts they never will. If you need order-preserving deduplication, use `dict.fromkeys()`.
3. **Trying to index a set.** `s[0]` raises `TypeError: 'set' object is not subscriptable`. Convert to a list first, or use a different structure.
4. **Putting a list in a set.** Elements must be hashable. Use a `tuple` (or `frozenset`) instead.
5. **Confusing `remove()` with `discard()`.** `remove()` raises `KeyError` if absent; `discard()` silently does nothing.
6. **Confusing `union()` with `update()`.** `union()` returns a new set; `update()` mutates in place and returns `None`.
7. **Assuming `pop()` removes the 'first' element.** Sets have no order — `pop()` removes an *arbitrary* element.
8. **Using `set` when duplicates carry meaning.** Deduplicating a shopping list silently loses the fact that you wanted two litres of milk.

## Best Practices

- Reach for a set whenever the question is *'have I seen this?'* or *'what's in both?'*
- Use `set()` for the empty set; use `{...}` only when it has elements.
- Use the operators (`|`, `&`, `-`, `^`) when both sides are sets; use the method forms (`union`, `intersection`) when the other side is any iterable.
- Use `discard()` when absence is fine, `remove()` when absence is a bug you want to hear about.
- Use `frozenset` when you need a set as a dict key or inside another set.
- Use `dict.fromkeys(seq)` to deduplicate while preserving order.
- Convert a list to a set *before* a loop of membership tests, not inside it.

## Practice Exercises

Try these before moving on.

1. Find the characters that appear in both `"python"` and `"typhoon"`, and those in only one.
2. Given two lists of student names, find who is in both classes, who is only in the first, and the full roster with no duplicates.
3. Deduplicate `[3,1,2,3,1]` twice: once with `set()` and once preserving order. Compare results.
4. Explain why `{[1,2]}` raises `TypeError` but `{(1,2)}` does not.
5. Build a set of `frozenset`s representing unordered pairs, so that `{a,b}` and `{b,a}` count as the same pair.
6. Time `x in big_list` against `x in big_set` for 100,000 lookups.
7. Use a set comprehension to collect the distinct word lengths in a sentence.
8. Given `a = {1,2,3}` and `b = {3,4,5}`, produce elements in exactly one of them — two ways.